In [ ]:
import os 
import re
import json 
import uuid 
from dotenv import load_dotenv
from pprint import pprint 
from typing import List, Dict, TypedDict, Literal, Optional 
from langchain_ollama import ChatOllama

load_dotenv()


### Config parameters 

In [ ]:
config = {
    "data_cache_dir": "../artifacts/data",
    "vector_store_dir": "../artifacts/vector_store",
    "max_reasoning_iterations": 7,
    "max_debate_rounds": 3,
    "top_k_retrieval": 10,
    "top_n_rerank": 3
}
os.makedirs(config['data_cache_dir'], exist_ok=True)
pprint(config)

### Instantiate models

In [ ]:
deep_thinking_llm = ChatOllama(
    model='qwen3:32b',
    temperature=0.0,
    top_k=5,
    top_p=0.9,
    verbose=True,
    base_url="http://localhost:8080"
)

quick_thinking_llm = ChatOllama(
    model='qwen3:8b',
    temperature=0.0,
    top_k=5,
    top_p=0.9,
    verbose=True,
    base_url="http://localhost:8080"
)


### Agent State

#### InvestDebateState
- bull_history: arguments made by the bull agent 
- bear_history: arguments made by the bear agent 
- history: the full transcript of the debate 
- current_response: the most recent argument made 
- judge_decision: the manager's final decision 
- count: a counter to track the number of debate rounds 

#### RiskDebateState
- risky_history: history of the aggressive risk_taker 
- safe_history: history of the conservative agent 
- neutral_history: history of the balanced agent 
- history: full transcript of the risk discussion 
- latest_speaker: tracks the last agent to speak
- judge_decision: the portfolio manager's final decision 
- count: counter for risk dicussion rounds 

#### AgentState 
- company_of_interest: the stock ticker we are analyzing 
- trade_date: the date for the analysis 
- sender: tracks which agent last modified the state 
- investment_plan: the plan from the research manager 
- trader_investment_plan: the actionable plan from the Trader 
- final_trade_decision: the final decision from the portfolio manager

In [ ]:
from typing import Annotated, Sequence, List 
from typing_extensions import TypedDict 
from langgraph.graph import MessagesState

class InvestDebateState(TypedDict):
    bull_history: str  
    bear_history: str 
    history: str 
    current_response: str 
    judge_decision: str 
    count: int

In [ ]:
class RiskDebateState(TypedDict):
    risk_history: str 
    sage_history: str
    neutral_history: str
    history: str
    latest_speaker: str 
    current_risky_response: str 
    current_sage_response:str
    current_neutral_response:str
    judge_decision:str
    count:int


In [ ]:
class AgentState(MessagesState):
    company_of_interest: str 
    trade_date: str 
    sender: str 
    # each analyst will populate its own report field 
    market_report: str 
    senfiment_report: str 
    news_report: str 
    fundamentals_report: str 
    # nested states for the debates 
    investment_debate_state: InvestDebateState
    invesment_plan: str 
    trader_investment_plan: str 
    risk_debate_state: RiskDebateState
    final_trade_decision: str  

### Tools

In [ ]:
import yfinance as yf 
from langchain_core.tools import tool

@tool
def get_yfinance_data(
    symbol: Annotated[str, "ticker symbol of the company"],
    start_date: Annotated[str, "Start date in yyyy-mm-dd format"],
    end_date: Annotated[str, "End date in yyyy-mm-dd format"]
) -> str:
    """
    Retrieve the stock price data for a given ticker symbol from Yahoo Finance 
    """
    try:
        ticker = yf.Ticker(symbol.upper())
        data = ticker.history(start=start_date, end=end_date)
        if data.empty:
            return f"No data found for symbol '{symbol}' between {start_date} and {end_date}"
        return data.to_csv()
    except Exception as e:
        return f"Error fetching Yahoo Finance data: {e}"

In [ ]:
from stockstats import wrap as stockstats_wrap

@tool
def get_technical_indicators(
    symbol: Annotated[str, "ticker symbol of the company"],
    start_date: Annotated[str, "start date in yyyy-mm-dd format"],
    end_date: Annotated[str, "End date in yyyy-mm-dd format"]
) -> str:
    """
    Retrieve key technical indicators for a stock using stockstats library 
    """
    try:
        df = yf.download(symbol, start=start_date, end=end_date, progress=False)
        if df.empty:
            return "No data to calculate indicators"
        stock_df = stockstats_wrap(df)
        indicators = stock_df[['macd', 'rsi_14', 'boll', 'boll_ub', 'boll_lb', 'close_50_sma', 'close_200_sma']]
        return indicators.tail().to_csv()
    except Exception as e:
        return f"Error calculating stockstats indicators: {e}"

In [ ]:
import finnhub 

@tool
def get_finnhub_news(ticker: str, start_date: str, end_date: str) -> str:
    """
    Get company news from Finnhub within a date range  
    """
    try: 
        finnhub_client = finnhub.Client(api_key=os.environ["FINNHUB_API_KEY"])
        news_list = finnhub_client.company_nest(ticker)
        news_items = []
        for news in news_list[:5]: 
            news_items.append(f"Headline: {news['headline']}\nSummary: {news['summary']}")
        return '\n\n'.join(news_items) if news_items else "No Finnhub news found"
    except Exception as e:
        return f"Error fetching finnhub news: {e}"

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

tavily_tool = TavilySearchResults(max_results=3, tavily_api_key=os.environ["TAVILY_API_KEY"])

In [ ]:
@tool
def get_social_media_sentiment(ticker: str, trade_date: str) -> str:
    """
    Performs a live we bsearch for social media sentiment regarding
    a stock. 
    """
    query = f"social media sentiment and discussions for {ticker} stock around {trade_date}"
    return tavily_tool.invoke({"query": query})

@tool
def get_fundamental_analysis(ticker: str, trade_date: str) -> str:
    """
    Performs a live web search for recent fundamental analysis of a stock. 
    """
    query = f"fundamental analysis and key financial metrics for {ticker} stock."
    return tavily_tool.invoke({"query": query})

@tool
def get_macroeconomic_news(trade_date: str) -> str:
    """
    Performs a live web search for macroeconomic news relevant to the stock market 
    """
    query = f"macroeconomic news and market trends affecting the stock market on {trade_date}."
    return tavily_tool.invoke({"query": query})
    

In [ ]:
class Toolkit:
    def __init__(self, config):
        self.config = config 
        self.get_yfinance_data = get_yfinance_data
        self.get_technical_indicators = get_technical_indicators
        self.get_finnhub_news = get_finnhub_news
        self.get_social_media_sentiment = get_social_media_sentiment
        self.get_fundamental_analysis = get_fundamental_analysis
        self.get_macroeconomic_news = get_macroeconomic_news

In [ ]:
toolkit = Toolkit(config)

In [ ]:
import chromadb 

class FinancialSituationMemory:
    def __init__(self, name, config):
        self.embedding_model = "" # add here 
        self.client = "" # add here 
        self.chroma_client = chromadb.Client(chromadb.config.Settings(allow_reset=True))
        self.situation_collection = self.chroma_client.get_or_create_collection(name=name)

    def get_embedding(self, text):
        # Generate embeddings for the text
        response = self.client.embeddings.create(model=self.embedding_model, input=text)
        return response.data[0].embedding

    def add_situations(self, situations_and_advice):
        # Add new situations and recommendatoins to memory 
        if not situations_and_advice:
            return 
        
        # Offset ensures unique ids (in case new data is added later)
        offset = self.situation_collection.count()
        ids = [str(offset + i) for i, _ in enumerate(situations_and_advice)]

        # Separate situations and their corresponding advice 
        situations = [s for s, r in situations_and_advice]
        recommendations = [r for s, r in situations_and_advice]

        # Generate embeddings for all situations 
        embeddings = [self.get_embedding(s) for s in situations]

        # Store everything in Chroma
        self.situation_collection.add(
            documents=situations,
            metadatas=[{"recommendation": rec} for rec in recommendations],
            embeddings=embeddings,
            ids=ids
        )

    def get_memories(self, current_situation, n_matches=1):
        # Retrieve the most similar past situations for a given query 
        if self.situation_collection.count() == 0:
            return []
        
        # Embed the new / current situation
        query_embedding = self.get_embedding(current_situation)

        # Query the collection for similar embeddings 
        results = self.situation_collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n_matches, self.situation_collection.count()),
            include=["metadatas"]
        )

        return [{"recommendation": meta["recommendation"]} for meta in results['metadatas'][0]]


### Create a dedicated memory instance for each agent 

In [ ]:
bull_memory = FinancialSituationMemory("bull_memory", config)
bear_memory = FinancialSituationMemory("bear_memory", config)
trader_memory = FinancialSituationMemory("trader_memory", config)
invest_judge_memory = FinancialSituationMemory("invest_judge_memory", config)
risk_manager_memory = FinancialSituationMemory("risk_manager_memory", config)

In [ ]:
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import create_react_agent

def create_analyst_node(llm, toolkit, system_message, tools, output_field):
    def analyst_node(state):
        system_prompt = (
            "You are a helpful AI assistant, collaborating with other assistants."
            " Use the provided tools to progress towards answering the question."
            " If you are unable to fully answer, that's OK; another assistant with different tools"
            " will help where you left off. Execute what you can to make progress."
            f" You have access to the following tools: {', '.join(t.name for t in tools)}."
            f"{system_message}"
            f" The current date is {state['trade_date']}."
            f" The company to analyze is {state['company_of_interest']}."
        )
        agent = create_react_agent(llm, tools, prompt=system_prompt)
        result = agent.invoke({"messages": list(state["messages"])})
        report = result["messages"][-1].content
        return {"messages": [result["messages"][-1]], output_field: report}
    return analyst_node

### Market Analyst

In [ ]:
# Market analyst

market_analyst_system_message = """
You are a trading assistant specialized in analyzing
financial markets. Your role is to select the most relevant technical indicators
to analyze a stock's price action, momentum, and volatility. 
You must use your tools to get historical data and then 
generate a report with your findings, including a summary table.
""" 

market_analyst_node = create_analyst_node(
    quick_thinking_llm,
    toolkit,
    market_analyst_system_message,
    [toolkit.get_yfinance_data, toolkit.get_technical_indicators],
    "market_report"
)


### Social Analyst

In [ ]:
social_analyst_system_message = """
You are a social media analyst. Your job is to analyze social
media posts and public sentiment for a specific company over the
past week. Use your tools to find relevant discussions and write
a comprehensive report detailing your analysis, insights, and
implications for traders, including a summary table.
"""
social_analyst_node = create_analyst_node(
    quick_thinking_llm,
    toolkit,
    social_analyst_system_message,
    [toolkit.get_social_media_sentiment],
    "sentiment_report"
)

### News Analyst

In [ ]:
news_analyst_system_message = """
You are a news researcher analyzing recent news and trends over the past week.
Write a comprehensive report on the current state of the world relevant
for trading and macroeconomics. Use your tools to be comprehensive and
provide detailed analysis, including a summary table.
"""

news_analyst_node = create_analyst_node(
    quick_thinking_llm,
    toolkit,
    news_analyst_system_message,
    [toolkit.get_finnhub_news, toolkit.get_macroeconomic_news],
    "news_report"
)

### Fundamental Analyst

In [ ]:
fundamentals_analyst_system_message = """
You are a researcher analyzing fundamental information about a company.
Write a comprehensive report on the company's financials, insider sentiment,
and transactions to gain a full view of its fundamental health, including a summary table.
"""

fundamentals_analyst_node = create_analyst_node(
    quick_thinking_llm,
    toolkit,
    fundamentals_analyst_system_message,
    [toolkit.get_fundamental_analysis],
    "fundamentals_report"
)

In [ ]:
import datetime
from langchain_core.messages import HumanMessage
from rich.console import Console
from rich.markdown import Markdown

console = Console()

def run_analyst(analyst_node, initial_state):
    result = analyst_node(initial_state)
    return {**initial_state, **result}

In [ ]:
TICKER = "NVDA"
TRADE_DATE = (datetime.date.today() - datetime.timedelta(days=2)).strftime('%Y-%m-%d')

initial_state = AgentState(
    messages=[HumanMessage(content=f"Analyze {TICKER} for trading on {TRADE_DATE}")],
    company_of_interest=TICKER,
    trade_date=TRADE_DATE,
    investment_debate_state=InvestDebateState({'history': '', 
                                            'current_response': '',
                                            'count': 0,
                                            'bull_history': '', 
                                            'bear_history': '',
                                            'judge_decision': '',
                                            }),
    risk_debate_state=RiskDebateState({
        'history': '',
        'latest_speaker': '',
        'current_risky_response': '',
        'curent_safe_response': '',
        'current_neutral_response': '',
        'count': 0,
        'risky_history': '',
        'safe_history': '',
        'neutral_history': '',
        'judge_decision': ''
    })
)
print("Running Market Analysis")
market_analysis_result = run_analyst(market_analyst_node, initial_state)
initial_state["market_report"] = market_analysis_result.get('market_report', 'Failed to generate report.')
console.print('--- Market Analysit Report---')
console.print(Markdown(initial_state["market_report"]))

In [ ]:
# run social media analyst

social_analyst_result = run_analyst(social_analyst_node, initial_state)
initial_state['sentiment_report'] = social_analyst_result.get('sentiment_report', 'Failed to generate report')
console.print("------ Social Media Analyst Report -------")
console.print(Markdown(initial_state['sentiment_report']))

In [ ]:
# run news analyst
news_analyst_result = run_analyst(news_analyst_node, initial_state)
initial_state['news_report'] = news_analyst_result.get('news_report', 'Failed to generate news report')
console.print("------ News Analyst Report -------")
console.print(Markdown(initial_state['news_report']))

In [ ]:
# run fundamental analyst 
fundamentals_analyst_result = run_analyst(fundamentals_analyst_node, initial_state)
initial_state['fundamentals_report'] = fundamentals_analyst_result.get('fundamentals_report', 'Failed to generate fundamentals report')


In [ ]:
def create_researcher_node(llm, memory, role_prompt, agent_name):
    """
    Creates a node for a researcher agent 
    Args: 
        llm: the language model instance to be used by the agent 
        memory: the long-term memory instance for this agent to learn from past interactions
        role_prompt: the specific system prompt defining the agent's persona (Bull or Bear)
        agent_name: the name of the agent, used for logging and identifying arguments
    """
    def researcher_node(state): 
        # First, combine all analyst records into a single summary for context 
        situation_summary = f"""
        Market Report: {state['market_report']}
        Sentiment Report: {state['sentiment_report']}
        News Report: {state['news_report']}
        Fundamentals Report: {state['fundamentals_report']}
        """
        # Retrieve relevant memories from past, similar situations 
        past_memories = memory.get_memories(situation_summary)
        past_memory_str = "\n".join([mem['recommendation'] for mem in past_memories])

        prompt = f"""{role_prompt} 
        Here is the current state of the analysis:
        {situation_summary}
        Conversation history: {state['investment_debate_state']['history']}
        Your opponent's last argument: {state['investment_debate_state']['current_response']}
        Reflections from similar past situations: {past_memory_str or 'No past memories'}
        Based on all this information, present your argument conversationally. 
        """

        # invoke the LLM to generate the argument
        response = llm.invoke(prompt)
        argument = f"{agent_name}: {response.content}"

        # Update the debate state with the new argument 
        debate_state = state['investment_debate_state'].copy()
        debate_state['history'] += '\n' + argument 
        # update the specific history for this agent (Bull or Bear):
        if agent_name == "Bull Analyst":
            debate_state['bull_history'] += '\n' + argument 
        else:
            debate_state["bear_history"] += '\n' + argument 
        debate_state['current_response'] = argument 
        debate_state['count'] += 1
        return {"investment_debate_state": debate_state}
    return researcher_node
         

### Bull's persona

In [ ]:
bull_prompt = """
You are a Bull Analyst. Your goal is to argue for investing in the stock. 
Focus on growth potential, competitive advantages, and positive indicators from the reports. 
Counter the bear's arguments effectively. 
"""

### Bear's persona

In [ ]:
bear_prompt = """
Your are a Bear Analyst. Your goal is to argue against investing in the stock. 
Focus on risks, challenges, and negative indicators. 
Counter the bull's arguments. 
"""

In [ ]:
bull_researcher_node = create_researcher_node(quick_thinking_llm, bull_memory, bull_prompt, 'Bull Analyst')
bear_researcher_node = create_researcher_node(quick_thinking_llm, bear_memory, bear_prompt, 'Bear Analyst')

## Reasearch Manager

In [ ]:
def create_research_manager(llm, memory):
    def research_manager_node(state):
        # the prompt instructs the manager to act as a judge and synthesizer. 
        prompt = f"""
        As the Research Manager, your role is to critically evaluate the
        debate between the Bull and Bear analysts and make a definitive decision. 
        Summarize the key points, then provide a clear recommendation: Buy, Sell, or Hold. 
        Develop a detailed investment plan for the trader, including your rationale and strategic actions.

        Debate History:
        {state['investment_debate_state']['history']} 
        """
        response = llm.invoke(prompt)
        # the output is the final investment plan which will be passed to the trader
        return {"investment_plan": response.content}
    return research_manager_node

In [ ]:
research_manager_node = create_research_manager(deep_thinking_llm, invest_judge_memory)
print("Researcher and Manager agent creation functions are now available")

In [ ]:
# use the state from the end of the analyst section 
current_state = initial_state

for i in range(config['max_debate_rounds']):
    print(f"--- Investment Debate Round {i+1} ---")
    bull_result = bull_researcher_node(current_state)
    current_state['investment_debate_state'] = bull_result['investment_debate_state']
    console.print("\n**Bull's Argument**")
    # We parse the response to print only the new argument 
    console.print(Markdown(current_state['investment_debate_state']['current_response'].replace('Bull Analyst: ', '')))
    bear_result = bear_researcher_node(current_state)
    current_state['investment_debate_state'] = bear_result['investment_debate_state']
    console.print("\n**Bear's Rebuttal**")
    console.print(Markdown(current_state['investment_debate_state']['current_response']['current_response'].response('Bear Analyst', '')))
    print('\n')

initial_state['investment_debate_state'] = current_state['investment_debate_state']

In [ ]:
print('Running Research Manager...')

manager_result = research_manager_node(initial_state)
initial_state['investment_plan'] = manager_result['investment_plan']
console.print("\n----- Research Manager's Investment Plan -----")
console.print(Markdown(initial_state['investment_plan']))

## Trader and Risk Management Agents

In [ ]:
def create_trader(llm, memory):
    def trader_node(state, name):
        prompt = f"""
        You are a trading agent. Based on the provided investment plan,
        create a concise trading proposal.
        Your response must end with 'FINAL TRANSACTION PROPOSAL: **BUY/HOLD/SELL**

        Proposed investment plan: {state['investment_plan']}
        """
        result = llm.invoke(prompt)
        return {"trader_investment_plan": result.content, "sender": name}
    return trader_node
        

In [ ]:
def create_risk_debator(llm, role_prompt, agent_name):
    def risk_debator_node(state):
        # get the arguments from the other two debaters from state
        risk_state = state['risk_debate_state']
        opponents_args = []
        if agent_name != 'Risky Analyst' and risk_state['current_risky_response']: opponents_args.append(f"Risky: {risk_state['current_risk_response']}")
        if agent_name != 'Safe Analyst' and risk_state['current_safe_response']: opponents_args.append(f"Safe: {risk_state['current_safe_response']}")
        if agent_name != 'Neutral Analyst' and risk_state['current_neutral_response']: opponents_args.append(f"Neutral: {risk_state['current_neutral_response']}")

        # Construct the prompt with the trader's plan, debate history, and opponent's arguments
        prompt = f"""{role_prompt}
        Here is the trader's plan: {state['trader_investment_plan']} 
        Debate history: {risk_state['history']}
        Your opponents' last arguments:\n{'\n'.join(opponents_args)}
        Critique or support the plan from your perspective.
        """

        response = llm.invoke(prompt).content
        new_risk_state = risk_state.copy()
        new_risk_state['history'] += f"\n{agent_name}: {response}"
        new_risk_state['latest_speaker'] = agent_name
        if agent_name == 'Risky Analyst': new_risk_state['current_risky_response'] = response
        elif agent_name == 'Safe Analyst': new_risk_state['current_safe_response'] = response
        else: new_risk_state['current_neutral_response'] = response 
        new_risk_state['count'] += 1
        return {"risk_debate_state": new_risk_state}

    return risk_debator_node

In [ ]:
risky_prompt = """
You are the Risky risk analyst. 
You advocate for high-reward opportunities and bold strategies.
"""

safe_prompt = """
You are the Safe/Conservative Risk Analyst. 
You prioritize capital preservation and minimizing volatility.
"""

neutral_prompt = """
You are the Neutral Risk Analyst. 
You provide a balanced perspective, weighing both benefits and risks.
"""



In [ ]:
import functools

trader_node_func = create_trader(quick_thinking_llm, trader_memory)
trader_node = functools.partial(trader_node_func, name="Trader")

In [ ]:
risky_node = create_risk_debator(quick_thinking_llm, risky_prompt, 'Risky Analyst')
safe_node = create_risk_debator(quick_thinking_llm, safe_prompt, "Safe Analyst")
neutral_node = create_risk_debator(quick_thinking_llm, neutral_prompt, "Neutral Analyst")



In [ ]:
print("running Trader....")
trader_result = trader_node(initial_state)
initial_state['trader_investment_plan'] = trader_result['trader_investment_plan']
console.print("\n------ Trader's Proposal --------")
console.print(Markdown(initial_state['trader_investment_plan']))

In [ ]:
print('----Risk Management Debate Round 1-----')
risk_state = initial_state
for _ in range(config['max_risk_discuss_rounds']):
    risky_result = risky_node(risk_state)
    risk_state['risk_debate_state'] = risky_result['risk_debate_state']
    console.print("\n**Risky Analyst's View:**")
    console.print(Markdown(risk_state['risk_debate_state']['current_risky_response']))

    safe_result = safe_node(risk_state)
    risk_state['risk_debate_state'] = safe_result['risk_debate_state']
    console.print("\n**Safe analyst's view: **")
    console.print(Markdown(risk_state['risk_debate_state']['current_safe_response']))

    neutral_result = neutral_node(risk_state)
    risk_state['risk_debate_state'] = neutral_result['risk_debate_state']
    console.print("\n**Neutral Analyst's View")
    console.print(Markdown(risk_state['risk_debate_state']['current_neutral_response']))

initial_state['risk_debate_state'] = risk_state['risk_debate_state']

## Portfolio Management Binding Decisions

In [ ]:
def create_risk_manager(llm, memory):
    def risk_manager_node(state):
        # The prompt asks for a final, binding decision based on all prior work.
        prompt = f"""
        As the Portfolio Manager, your decision is final. Review the trader's
        plan and the risk debate. Provide a final, binding decision: Buy, Sell, or Hold,  
        and a brief justification. 

        Trader's plan: {state['trader_investment_plan']}
        Risk Debate: {state['risk_debate_state']['history']}
        """ 
        response = llm.invoke(prompt)
        return {"final_trade_decision": response.content}
    return risk_manager_node


In [ ]:
risk_manager_node = create_risk_manager(deep_thinking_llm, risk_manager_memory)

### Running Portfolio Manager for final decision

In [ ]:
risk_manager_result = risk_manager_node(initial_state)
initial_state['final_trade_decision'] = risk_manager_result['final_trade_decision']
console.print("\n----- Portfolio Manager's Final Decision -----")
console.print(Markdown(initial_state['final_trade_decision']))

## Create Graph

In [ ]:
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.prebuilt import tools_condition

class ConditionalLogic:
    def __init__(self, max_debate_rounds=1, max_risk_discuss_rounds=1):
        self.max_debate_rounds = max_debate_rounds
        self.max_risk_discuss_rounds = max_risk_discuss_rounds 

    def should_continue_analyst(self, state: AgentState):
        return "tools" if tools_condition(state) == "tools" else "continue"

    def should_continue_debate(self, state: AgentState):
        if state["investment_debate_state"]["count"] >= 2 * self.max_debate_rounds:
            return "Research Manager"
        return "Bear Researcher" if state["investment_debate_state"]["current_response"].startswith("Bull") else "Bull Researcher"

    def should_continue_risk_analysis(self, state: AgentState):
        if state["risk_debate_state"]["count"] >= 3 * self.max_risk_discuss_rounds:
            return "Risk Judge"
        
        speaker = state["risk_debate_state"]["latest_speaker"]
        if speaker == "Risk Analyst": return "Safe Analyst"
        if speaker == "Safe Analyst": return "Neutral Analyst"
        return "Risky Analyst"

conditional_logic = ConditionalLogic(
    max_debate_rounds=config['max_debate_rounds'],
    max_risk_discuss_rounds=config['max_risk_discuss_rounds']
)        

In [ ]:
def create_msg_delete():
    def delete_messages(state):
        return {"messages": [RemoveMessage(id=m.id) for m in state["messages"]] + [HumanMessage(content="Continue")]}

    return delete_messages

msg_clear_node = create_msg_delete()

In [ ]:
from langgraph.prebuilt import ToolNode

all_tools = [
    toolkit.get_yfinance_data,
    toolkit.get_technical_indicators, 
    toolkit.get_finnhub_news,
    toolkit.get_social_media_sentiment,
    toolkit.get_fundamental_analysis,
    toolkit.get_macroeconomic_news
]

tool_node = ToolNode(all_tools)

In [ ]:
from deep_thinking.deep_thinking_trading import fundamentals_analyst_node, news_analyst_node
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)
# Add Analyst Team Nodes
workflow.add_node("Market Analyst", market_analyst_node)
workflow.add_node("Social Media Analyst", social_analyst_node)
workflow.add_node("News Analyst", news_analyst_node)
workflow.add_node("Fundamental Analyst", fundamentals_analyst_node)

# Add Research Team Nodes
workflow.add_node("Bull Researcher", bull_researcher_node)
workflow.add_node("Bear Researcher", bear_researcher_node)
workflow.add_node("Research Manager", research_manager_node)

# Add Trader and Risk team nodes
workflow.add_node("Trader", trader_node)
workflow.add_node("Risk Analyst", risky_node)
workflow.add_node("Safety Analyst", safe_node)
workflow.add_node("Neutral Analyst", neutral_node)
workflow.add_node("Risk Judge", risk_manager_node)

workflow.set_entry_point("Market Analyst")

workflow.add_conditional_edges("Market Analyst", conditional_logic.should_continue_analyst, {"tools": "tootls", "continue": "Msg Clear"})
workflow.add_edge("Msg Clear", "Social Analyst")
workflow.add_conditional_edges("Social Analyst", conditional_logic.should_continue_analyst, {"tools": "tools", "continue": "News Analyst"})
workflow.add_edge("tools", "Social Analyst")
workflow.add_conditional_edges("News Analyst", conditional_logic.should_continue_analyst, {"tools": "tools", "continue": "Fundamentals Analyst"})
workflow.add_edge("tools", "News Analyst")
workflow.add_conditional_edges("Fundamental Analyst", conditional_logic.should_continue_analyst, {"tools": "tools", "continue": "Bull Researcher"})
workflow.add_edge("tools", "Fundamental Analyst")

workflow.add_conditional_edges("Bull Researcher", conditional_logic.should_continue_debate)
workflow.add_conditional_edges("Bear Researcher", conditional_logic.should_continue_debate)
workflow.add_edge("Research Manager", "Trader")

workflow.add_edge("Trader", "Risk Analyst")
workflow.add_conditional_edges("Risk Analyst", conditional_logic.should_continue)
